In [ ]:
!pip install pinecone-client

In [ ]:
!pip install tiktoken
!pip install PyPDF2
!pip install langchain-pinecone pinecone

In [ ]:
!pip install langchain

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
pip install -U langchain-text-splitters

In [ ]:
!pip install langchain_google_genai

In [2]:
from PyPDF2 import PdfReader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

def extract_text_from_pdf(file):
    reader = PdfReader(file)
    text = ""

    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )

    chunks=text_splitter.split_text(text)

    return chunks

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [3]:
chunks=extract_text_from_pdf(r"C:\Users\Rudraksh\Summarizer\pdf\attention.pdf")

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector = embeddings.embed_query(chunks)
print(len(vector))


3072


In [5]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

In [6]:
index_name='test'
index=pc.Index(index_name)

# Create Embedding for each of the text Chunk

In [7]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_texts(
    texts=chunks,
    embedding=embeddings,
    index_name=index_name
)

In [68]:
docsearch

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [19]:
retriever = docsearch.as_retriever(
    search_kwargs={"k": 3}
)

In [20]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question:
{question}
""")

In [21]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()   #Runnable passthrough means return the input unchnged
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
while True:
    question = input("Question: ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nAnswer:")
    print(answer)
    print()


Answer:
The Transformer is a new simple network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.


Answer:
An attention function maps a query and a set of key-value pairs to an output. The query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.



In [6]:
import logging
import os
from datetime import datetime

LOG_FILE = f'{datetime.now().strftime('%m_%d_%Y_%H_%S')}.log'

In [7]:
print(LOG_FILE)

06_22_2026_16_45.log
